# Day 0 — "Same image, different answer": does batching change detection results?

**Question.** When an object detector processes images in a batch, each image is padded up to the size of the largest image in that batch. Does the prediction for an image change depending on *which other images* share its batch? And does that move the benchmark number (COCO AP)?

**Setup.**
- 500 random COCO val2017 images (seed 0).
- Three detectors:
  - **DETR-R50** (Hugging Face) — pads to the batch's largest image.
  - **Faster R-CNN v2** (torchvision) — pads to the batch's largest image.
  - **RT-DETR-R50** (Hugging Face) — the **control**. It resizes every image to 640×640, so there is no padding. Any change here comes only from GPU arithmetic, not from padding.

Every model sees the same 500 images under each batching condition:

| Condition | How the images are grouped |
|---|---|
| `bs1` | one image at a time (reference) |
| `bs1_repeat` | the same again (determinism check) |
| `bs8_rand_s{0,1,2}` | random groups of 8, three different shuffles |
| `bs8_sorted` | groups of 8 with similar shapes (least padding) |
| `bs8_mixed` | groups of 8 mixing tall and wide images (most padding) |

**What we measure against `bs1`:**
- **flip rate:** share of images whose confident detections (score ≥ 0.5) change — a box appears or disappears, or a matched box moves (IoU < 0.9);
- **ΔAP:** the change in COCO box AP on the 500 images;
- **seed range:** how far AP moves just by re-shuffling the random batches.

**Decision rule (fixed before looking at data):**
- **START** if any padding model shows ≥ 2% flips under random batching, OR |ΔAP| ≥ 0.3, OR an AP range ≥ 0.2 across the three shuffles.
- **KILL** if every model shows < 1% flips and |ΔAP| < 0.1.
- Anything in between is **AMBER**: we look closer before deciding.

**Kaggle settings:** Accelerator **GPU T4 x2** or **P100** (check that the second code cell prints a GPU name!). Internet **ON**. There is no need to attach a dataset: the notebook downloads the COCO annotations (~250 MB) and only the 500 images it needs. If you already attached a COCO 2017 dataset, it uses that instead. Expected runtime: roughly 30–45 minutes.

In [ ]:
!pip install -q "transformers==5.17.0" pycocotools

In [ ]:
import os, io, sys, glob, json, time, math, random, zipfile, platform, datetime, contextlib, urllib.request
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd, torch, torchvision
import matplotlib.pyplot as plt
from PIL import Image
import transformers
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

OUT = "/kaggle/working/batchdep_day0"
os.makedirs(OUT, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# Default GPU settings on purpose: we measure what a normal user would get.
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

ENV = {
    "run_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
    "python": platform.python_version(), "torch": torch.__version__, "torchvision": torchvision.__version__,
    "transformers": transformers.__version__,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
}
print(json.dumps(ENV, indent=2))
if ENV["gpu"] is None:
    print("\n⚠⚠ NO GPU. Stop, set Accelerator to GPU in the right panel, and Run All again. ⚠⚠")

## 1. Configuration

In [ ]:
CFG = {
    "n_images": 500,
    "seed": 0,
    "batch_size": 8,
    "random_seeds": [0, 1, 2],
    "models": ["detr_r50", "frcnn_v2", "rtdetr_r50"],
    "control_models": ["rtdetr_r50"],          # fixed-size resize, no padding
    "flip_score_thr": 0.5,
    "flip_iou": 0.9,
    "coco_root": "/kaggle/input",
    "coco_download_dir": "/kaggle/working/coco",
    "gate": {"flip_start": 0.02, "dap_start": 0.3, "seed_range_start": 0.2,
             "flip_kill": 0.01, "dap_kill": 0.1},
}
random.seed(CFG["seed"]); np.random.seed(CFG["seed"]); torch.manual_seed(CFG["seed"])

## 2. COCO val2017: annotations + 500 images

In [ ]:
def find_attached_coco(root):
    anns = glob.glob(os.path.join(root, "**", "instances_val2017.json"), recursive=True)
    probe = glob.glob(os.path.join(root, "**", "000000000139.jpg"), recursive=True)
    if anns and probe:
        return anns[0], os.path.dirname(probe[0])
    return None, None

ANN, IMG_DIR = find_attached_coco(CFG["coco_root"])
if ANN is None:
    os.makedirs(CFG["coco_download_dir"], exist_ok=True)
    ANN = os.path.join(CFG["coco_download_dir"], "instances_val2017.json")
    IMG_DIR = os.path.join(CFG["coco_download_dir"], "val2017")
    if not os.path.exists(ANN):
        zp = os.path.join(CFG["coco_download_dir"], "ann.zip")
        print("downloading COCO annotations (~250 MB) ...")
        urllib.request.urlretrieve("http://images.cocodataset.org/annotations/annotations_trainval2017.zip", zp)
        with zipfile.ZipFile(zp) as z, open(ANN, "wb") as f:
            f.write(z.read("annotations/instances_val2017.json"))
        os.remove(zp)
print("annotations:", ANN)

with contextlib.redirect_stdout(io.StringIO()):
    GT = COCO(ANN)
NAME2CAT = {c["name"].strip().lower(): c["id"] for c in GT.loadCats(GT.getCatIds())}
all_ids = sorted(GT.getImgIds())
IDS = sorted(random.Random(CFG["seed"]).sample(all_ids, CFG["n_images"]))
INFO = GT.loadImgs(IDS)

os.makedirs(IMG_DIR, exist_ok=True)
def fetch(info):
    p = os.path.join(IMG_DIR, info["file_name"])
    if not os.path.exists(p):
        urllib.request.urlretrieve(f"http://images.cocodataset.org/val2017/{info['file_name']}", p)
    return p
with ThreadPoolExecutor(16) as ex:
    paths = list(ex.map(fetch, INFO))
IMGS = [Image.open(p).convert("RGB") for p in paths]
N = len(IMGS)
aspect = np.array([im.size[0] / im.size[1] for im in IMGS])
print(f"{N} images loaded; aspect ratio w/h: min {aspect.min():.2f}, max {aspect.max():.2f}, "
      f"portrait {np.mean(aspect < 1):.0%}")

## 3. Batching conditions

Each condition is a partition of the 500 images into batches. Every image appears exactly once per condition, so each condition is a full evaluation run.

In [ ]:
B = CFG["batch_size"]
def chunks(order):
    return [list(order[i:i + B]) for i in range(0, len(order), B)]

CONDITIONS = {"bs1": [[i] for i in range(N)], "bs1_repeat": [[i] for i in range(N)]}
for s in CFG["random_seeds"]:
    CONDITIONS[f"bs8_rand_s{s}"] = chunks(np.random.default_rng(s).permutation(N))
by_shape = np.lexsort((np.array([im.size[0] * im.size[1] for im in IMGS]), aspect))
CONDITIONS["bs8_sorted"] = chunks(by_shape)
lo, hi, mixed = 0, N - 1, []
while lo <= hi:                      # alternate the tallest and the widest remaining image
    mixed.append(by_shape[lo]); lo += 1
    if lo <= hi:
        mixed.append(by_shape[hi]); hi -= 1
CONDITIONS["bs8_mixed"] = chunks(np.array(mixed))
for k, v in CONDITIONS.items():
    assert sorted(i for b in v for i in b) == list(range(N)), k
print({k: len(v) for k, v in CONDITIONS.items()})

## 4. Detectors

Each wrapper takes a list of images and returns, per image: boxes (x1, y1, x2, y2 in original pixels), scores, COCO category ids, and the **padding fraction** (the share of the model's input canvas that is padding for that image). Labels are mapped to COCO ids by class name, and the cell reports any name that fails to map.

In [ ]:
def _names_to_cat(names):
    m = {}
    for k, v in names.items():
        c = NAME2CAT.get(str(v).strip().lower())
        if c is not None:
            m[int(k)] = c
    return m

class HFDetector:
    def __init__(self, repo, cls_name, revision=None):
        from transformers import AutoImageProcessor
        cls = getattr(transformers, cls_name)
        self.proc = AutoImageProcessor.from_pretrained(repo, revision=revision)
        try:
            model = cls.from_pretrained(repo, revision=revision)
        except Exception as ex:
            # Some older checkpoints store `null` for fields that newer transformers requires to be set
            # (e.g. DETR's "dilation"). Drop those nulls so the library defaults are used, then retry.
            if "alidation" not in f"{type(ex).__name__} {ex}":
                raise
            import dataclasses
            from huggingface_hub import hf_hub_download
            raw = json.load(open(hf_hub_download(repo, "config.json", revision=revision)))
            defaults = {f.name: f.default for f in dataclasses.fields(cls.config_class)
                        if f.default is not dataclasses.MISSING}
            dropped = [k for k, v in raw.items() if v is None and defaults.get(k) is not None]
            print(f"  (config fix for {repo}: replaced null fields {dropped} with library defaults)")
            raw = {k: v for k, v in raw.items() if k not in dropped}
            model = cls.from_pretrained(repo, revision=revision, config=cls.config_class.from_dict(raw))
        self.model = model.to(DEVICE).eval()
        self.lab2cat = _names_to_cat(self.model.config.id2label)
        self.name = repo

    @torch.inference_mode()
    def __call__(self, imgs):
        enc = self.proc(images=imgs, return_tensors="pt")
        kw = {"pixel_values": enc["pixel_values"].to(DEVICE)}
        if "pixel_mask" in enc:
            kw["pixel_mask"] = enc["pixel_mask"].to(DEVICE)
        out = self.model(**kw)
        sizes = torch.tensor([[im.size[1], im.size[0]] for im in imgs])
        res = self.proc.post_process_object_detection(out, threshold=0.0, target_sizes=sizes)
        if "pixel_mask" in enc:
            pad = (1 - enc["pixel_mask"].float().mean(dim=(1, 2))).tolist()
        else:
            pad = [0.0] * len(imgs)
        return [self._fmt(r) for r in res], pad

    def _fmt(self, r):
        lab = r["labels"].cpu().numpy()
        keep = np.array([l in self.lab2cat for l in lab], bool)
        return {"boxes": r["boxes"].float().cpu().numpy()[keep], "scores": r["scores"].float().cpu().numpy()[keep],
                "labels": np.array([self.lab2cat[l] for l in lab[keep]], dtype=int)}

class TVFasterRCNN:
    def __init__(self):
        from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2, FasterRCNN_ResNet50_FPN_V2_Weights
        w = FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT
        self.model = fasterrcnn_resnet50_fpn_v2(weights=w).to(DEVICE).eval()
        self.tf = w.transforms()
        self.lab2cat = _names_to_cat(dict(enumerate(w.meta["categories"])))
        self.name = "torchvision/fasterrcnn_resnet50_fpn_v2"

    @torch.inference_mode()
    def __call__(self, imgs):
        x = [self.tf(im).to(DEVICE) for im in imgs]
        il, _ = self.model.transform(x)
        H, W = il.tensors.shape[-2:]
        pad = [1 - (h * w) / (H * W) for h, w in il.image_sizes]
        out = self.model(x)
        res = []
        for o in out:
            lab = o["labels"].cpu().numpy()
            keep = np.array([l in self.lab2cat for l in lab], bool)
            res.append({"boxes": o["boxes"].float().cpu().numpy()[keep], "scores": o["scores"].float().cpu().numpy()[keep],
                        "labels": np.array([self.lab2cat[l] for l in lab[keep]], dtype=int)})
        return res, pad

# Published COCO val2017 box AP (full 5k images) — our 500-image bs1 AP should land within ~±2 of these.
REF_AP = {"detr_r50": 42.0, "frcnn_v2": 46.7, "rtdetr_r50": 53.1}

LOADERS = {
    "detr_r50": lambda: HFDetector("facebook/detr-resnet-50", "DetrForObjectDetection", revision="no_timm"),
    "frcnn_v2": lambda: TVFasterRCNN(),
    "rtdetr_r50": lambda: HFDetector("PekingU/rtdetr_r50vd", "RTDetrForObjectDetection"),
}

## 5. Comparison helpers

In [ ]:
def iou_matrix(a, b):
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)))
    x1 = np.maximum(a[:, None, 0], b[None, :, 0]); y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2]); y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    area = lambda z: (z[:, 2] - z[:, 0]) * (z[:, 3] - z[:, 1])
    return inter / (area(a)[:, None] + area(b)[None, :] - inter + 1e-9)

def greedy_match(p, q, score_thr, iou_min):
    """Match p's detections (score>=thr) to q's (score>=thr), same class, highest score first."""
    ia = np.where(p["scores"] >= score_thr)[0]; ib = np.where(q["scores"] >= score_thr)[0]
    ia = ia[np.argsort(-p["scores"][ia])]
    M = iou_matrix(p["boxes"][ia], q["boxes"][ib]); used = set(); pairs = []
    for r, i in enumerate(ia):
        best, bj = -1, None
        for c, j in enumerate(ib):
            if c in used or p["labels"][i] != q["labels"][j]:
                continue
            if M[r, c] > best:
                best, bj = M[r, c], c
        if bj is not None and best >= iou_min:
            used.add(bj); pairs.append((i, ib[bj], best))
    return pairs, len(ia), len(ib)

def compare(p, q):
    pairs, na, nb = greedy_match(p, q, CFG["flip_score_thr"], 0.5)
    flip = (na != nb) or (len(pairs) != na) or any(iou < CFG["flip_iou"] for _, _, iou in pairs)
    allp, _, _ = greedy_match(p, q, 0.05, 0.5)
    dscore = max([abs(p["scores"][i] - q["scores"][j]) for i, j, _ in allp], default=0.0)
    return flip, dscore

def coco_ap(preds):
    dets = []
    for k, img_id in enumerate(IDS):
        r = preds[k]; order = np.argsort(-r["scores"])[:100]
        for t in order:
            x1, y1, x2, y2 = r["boxes"][t]
            dets.append({"image_id": img_id, "category_id": int(r["labels"][t]),
                         "bbox": [float(x1), float(y1), float(x2 - x1), float(y2 - y1)], "score": float(r["scores"][t])})
    with contextlib.redirect_stdout(io.StringIO()):
        dt = GT.loadRes(dets)
        E = COCOeval(GT, dt, "bbox"); E.params.imgIds = IDS
        E.evaluate(); E.accumulate(); E.summarize()
    return 100 * E.stats[0]

## 6. Run

For each model: load it, run every condition over all 500 images, then free the GPU before the next model.

In [ ]:
RESULTS, PER_IMAGE = [], []
for mname in CFG["models"]:
    t0 = time.time()
    det = LOADERS[mname]()
    print(f"\n== {mname} ({det.name}) loaded in {time.time() - t0:.0f}s; mapped classes: {len(set(det.lab2cat.values()))}/80")
    preds, pads = {}, {}
    for cname, batches in CONDITIONS.items():
        t1 = time.time(); P = [None] * N; PD = [None] * N
        for b in batches:
            r, p = det([IMGS[i] for i in b])
            for i, ri, pi in zip(b, r, p):
                P[i], PD[i] = ri, pi
        preds[cname], pads[cname] = P, PD
        ap = coco_ap(P)
        RESULTS.append({"model": mname, "condition": cname, "AP": ap, "mean_pad": float(np.mean(PD)),
                        "seconds": time.time() - t1})
        ref = f"   (published full-val AP ≈ {REF_AP[mname]}; a big gap means something is broken)" if cname == "bs1" else ""
        print(f"  {cname:14s} AP={ap:6.3f}  mean padding={np.mean(PD):.3f}  ({time.time() - t1:.0f}s){ref}")
    for cname in CONDITIONS:
        if cname == "bs1":
            continue
        for i in range(N):
            f, ds = compare(preds["bs1"][i], preds[cname][i])
            PER_IMAGE.append({"model": mname, "condition": cname, "image_id": IDS[i], "flip": f,
                              "max_score_delta": ds, "pad": pads[cname][i], "pad_bs1": pads["bs1"][i]})
    del det, preds; torch.cuda.empty_cache() if DEVICE == "cuda" else None

res = pd.DataFrame(RESULTS); per = pd.DataFrame(PER_IMAGE)
res.to_csv(f"{OUT}/ap_by_condition.csv", index=False); per.to_csv(f"{OUT}/per_image.csv", index=False)

## 7. Results

In [ ]:
base = res[res.condition == "bs1"].set_index("model")["AP"]
res["dAP_vs_bs1"] = res.apply(lambda r: r.AP - base[r.model], axis=1)
flip = per.groupby(["model", "condition"]).agg(flip_rate=("flip", "mean"),
                                               median_score_delta=("max_score_delta", "median"),
                                               p95_score_delta=("max_score_delta", lambda v: np.percentile(v, 95))).reset_index()
table = res.merge(flip, on=["model", "condition"], how="left")
display(table[["model", "condition", "AP", "dAP_vs_bs1", "flip_rate", "median_score_delta", "p95_score_delta", "mean_pad"]]
        .style.format({"AP": "{:.3f}", "dAP_vs_bs1": "{:+.3f}", "flip_rate": "{:.2%}", "median_score_delta": "{:.4f}",
                       "p95_score_delta": "{:.4f}", "mean_pad": "{:.3f}"}, na_rep="—"))

print("\nAP across the three random shuffles (batch 8):")
seed_stats = {}
for m in CFG["models"]:
    v = res[(res.model == m) & res.condition.str.startswith("bs8_rand")]["AP"].values
    seed_stats[m] = {"min": v.min(), "max": v.max(), "range": v.max() - v.min()}
    print(f"  {m:11s} {v.round(3).tolist()}  range = {v.max() - v.min():.3f}")

### Is it padding or just GPU arithmetic?

Images in a batch that happen to need **no padding** (padding = 0) can only change through arithmetic differences between batch sizes. Images **with** padding can also change through padding leaking into the features. If flips concentrate in padded images, padding is the cause.

In [ ]:
p8 = per[per.condition.str.startswith("bs8")].copy()
p8["pad_bin"] = pd.cut(p8["pad"], [-1e-9, 1e-6, 0.1, 0.25, 0.4, 1.0], labels=["0", "0–10%", "10–25%", "25–40%", ">40%"])
by_pad = p8.groupby(["model", "pad_bin"], observed=True).agg(flip_rate=("flip", "mean"), n=("flip", "size")).reset_index()
display(by_pad.pivot(index="pad_bin", columns="model", values="flip_rate").style.format("{:.2%}", na_rep="—"))
display(by_pad.pivot(index="pad_bin", columns="model", values="n"))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
conds = [c for c in CONDITIONS if c != "bs1"]
w = 0.8 / len(CFG["models"])
for k, m in enumerate(CFG["models"]):
    v = flip[flip.model == m].set_index("condition").reindex(conds)["flip_rate"] * 100
    axes[0].bar(np.arange(len(conds)) + (k - 1) * w, v, w, label=m)
axes[0].set_xticks(range(len(conds))); axes[0].set_xticklabels(conds, rotation=30, ha="right")
axes[0].set_ylabel("% images whose confident detections change vs bs1"); axes[0].legend()
axes[0].axhline(CFG["gate"]["flip_start"] * 100, ls="--", c="k", lw=1); axes[0].set_title("Flip rate by batching condition")
for m in CFG["models"]:
    d = by_pad[by_pad.model == m]
    axes[1].plot(d["pad_bin"].astype(str), d["flip_rate"] * 100, "o-", label=m)
axes[1].set_xlabel("padding fraction of the image in its batch"); axes[1].set_ylabel("flip rate (%)")
axes[1].set_title("Flip rate vs padding (batch-8 conditions)"); axes[1].legend()
plt.tight_layout(); plt.savefig(f"{OUT}/flip_rates.png", dpi=180); plt.show()

## 8. Gate

In [ ]:
G = CFG["gate"]
rows = []
for m in CFG["models"]:
    rand_flip = flip[(flip.model == m) & flip.condition.str.startswith("bs8_rand")]["flip_rate"].mean()
    max_dap = res[(res.model == m) & (res.condition != "bs1")]["dAP_vs_bs1"].abs().max()
    rows.append({"model": m, "control": m in CFG["control_models"], "random_flip_rate": rand_flip,
                 "max_abs_dAP": max_dap, "seed_AP_range": seed_stats[m]["range"],
                 "bs1_repeat_flip": flip[(flip.model == m) & (flip.condition == "bs1_repeat")]["flip_rate"].iloc[0]})
gate = pd.DataFrame(rows)
display(gate.style.format({"random_flip_rate": "{:.2%}", "max_abs_dAP": "{:.3f}", "seed_AP_range": "{:.3f}",
                           "bs1_repeat_flip": "{:.2%}"}))

pad_models = gate[~gate.control]
start = ((pad_models.random_flip_rate >= G["flip_start"]) | (pad_models.max_abs_dAP >= G["dap_start"]) |
         (pad_models.seed_AP_range >= G["seed_range_start"])).any()
kill = ((gate.random_flip_rate < G["flip_kill"]) & (gate.max_abs_dAP < G["dap_kill"])).all()
verdict = "START" if start else ("KILL" if kill else "AMBER")
print(f"\nVERDICT: {verdict}")
if (gate.bs1_repeat_flip > 0).any():
    print("Note: batch-1 runs are not fully repeatable on this GPU for some model(s); read flip rates relative to that floor.")
ctrl = gate[gate.control]
if len(ctrl) and (ctrl.random_flip_rate >= G["flip_kill"]).any():
    print("Note: the no-padding control also flips ≥1% — GPU arithmetic alone changes results; worth reporting separately.")

In [ ]:
manifest = {"env": ENV, "config": CFG, "image_ids": IDS, "verdict": verdict,
            "gate": gate.to_dict(orient="records"), "ap": res.to_dict(orient="records")}
with open(f"{OUT}/manifest.json", "w") as f:
    json.dump(manifest, f, indent=2, default=str)
print("saved:", sorted(os.listdir(OUT)))

## What to bring back

Upload the finished notebook here, like last time. The key outputs are the table in Section 7, the padding table, the figure, and the VERDICT line.